In [1]:
# =============================================================
#   ML Preprocessing Pipeline – Iris Dataset
#   Steps: Load → Inspect → Handle Missing → Encode → Scale → Split
# =============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

## 1. LOAD DATA ──────────────────────────────────────────────

In [2]:
df = pd.read_csv("/content/1) iris.csv")

print("=" * 55)
print("STEP 1: Raw Dataset")
print("=" * 55)
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nData Types:\n{df.dtypes}")

STEP 1: Raw Dataset
   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa

Shape: (150, 5)

Data Types:
sepal_length    float64
sepal_width     float64
petal_length    float64
petal_width     float64
species          object
dtype: object


## 2. INSPECT ────────────────────────────────────────────────

In [3]:
print("\n" + "=" * 55)
print("STEP 2: Dataset Info & Missing Values")
print("=" * 55)
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")


STEP 2: Dataset Info & Missing Values

Missing values per column:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

Basic Statistics:
       sepal_length  sepal_width  petal_length  petal_width
count    150.000000   150.000000    150.000000   150.000000
mean       5.843333     3.054000      3.758667     1.198667
std        0.828066     0.433594      1.764420     0.763161
min        4.300000     2.000000      1.000000     0.100000
25%        5.100000     2.800000      1.600000     0.300000
50%        5.800000     3.000000      4.350000     1.300000
75%        6.400000     3.300000      5.100000     1.800000
max        7.900000     4.400000      6.900000     2.500000


## 3. INJECT ARTIFICIAL MISSING VALUES (for demonstration) ───

In [4]:
# (Real Iris has none, so we simulate some to show the logic)
np.random.seed(42)
for col in ["sepal_length", "sepal_width", "petal_length", "petal_width"]:
    idx = np.random.choice(df.index, size=5, replace=False)
    df.loc[idx, col] = np.nan

print("\n" + "=" * 55)
print("STEP 3: After Injecting Missing Values (for demo)")
print("=" * 55)
print(f"Missing values:\n{df.isnull().sum()}")


STEP 3: After Injecting Missing Values (for demo)
Missing values:
sepal_length    5
sepal_width     5
petal_length    5
petal_width     5
species         0
dtype: int64


## 4. HANDLE MISSING DATA ────────────────────────────────────

In [5]:
print("\n" + "=" * 55)
print("STEP 4: Handling Missing Values")
print("=" * 55)

numerical_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

# Strategy: fill numerical NaNs with column MEAN
for col in numerical_cols:
    mean_val = df[col].mean()
    missing_count = df[col].isnull().sum()
    df[col] = df[col].fillna(mean_val)
    print(f"  '{col}': filled {missing_count} NaN(s) with mean = {mean_val:.4f}")

print(f"\nMissing values after handling:\n{df.isnull().sum()}")


STEP 4: Handling Missing Values
  'sepal_length': filled 5 NaN(s) with mean = 5.8221
  'sepal_width': filled 5 NaN(s) with mean = 3.0517
  'petal_length': filled 5 NaN(s) with mean = 3.7400
  'petal_width': filled 5 NaN(s) with mean = 1.1800

Missing values after handling:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64


## 5. ENCODE CATEGORICAL VARIABLE ────────────────────────────

In [6]:
print("\n" + "=" * 55)
print("STEP 5: Encoding Categorical Variable – 'species'")
print("=" * 55)

# Label Encoding  (setosa=0, versicolor=1, virginica=2)
le = LabelEncoder()
df["species_encoded"] = le.fit_transform(df["species"])

print(f"  Classes : {list(le.classes_)}")
print(f"  Mapping : { {cls: idx for idx, cls in enumerate(le.classes_)} }")
print(f"\nSample after encoding:\n{df[['species','species_encoded']].drop_duplicates()}")

# One-Hot Encoding (kept separately for reference)
df_ohe = pd.get_dummies(df["species"], prefix="species")
df = pd.concat([df, df_ohe], axis=1)
print(f"\nOne-Hot Encoded columns added: {list(df_ohe.columns)}")


STEP 5: Encoding Categorical Variable – 'species'
  Classes : ['setosa', 'versicolor', 'virginica']
  Mapping : {'setosa': 0, 'versicolor': 1, 'virginica': 2}

Sample after encoding:
        species  species_encoded
0        setosa                0
50   versicolor                1
100   virginica                2

One-Hot Encoded columns added: ['species_setosa', 'species_versicolor', 'species_virginica']


## 6. NORMALIZE / STANDARDIZE FEATURES ───────────────────────

In [7]:
print("\n" + "=" * 55)
print("STEP 6: Standardizing Numerical Features (Z-score)")
print("=" * 55)

X = df[numerical_cols].copy()
y = df["species_encoded"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=numerical_cols)

print("Before scaling (first 3 rows):")
print(df[numerical_cols].head(3).to_string(index=False))
print("\nAfter standardization (first 3 rows):")
print(X_scaled_df.head(3).to_string(index=False))
print(f"\nMean after scaling (should be ~0): {X_scaled_df.mean().round(4).tolist()}")
print(f"Std  after scaling (should be ~1): {X_scaled_df.std().round(4).tolist()}")


STEP 6: Standardizing Numerical Features (Z-score)
Before scaling (first 3 rows):
 sepal_length  sepal_width  petal_length  petal_width
          5.1          3.5           1.4          0.2
          4.9          3.0           1.4          0.2
          4.7          3.2           1.3          0.2

After standardization (first 3 rows):
 sepal_length  sepal_width  petal_length  petal_width
    -0.895023     1.040122     -1.352461     -1.30115
    -1.142928    -0.120014     -1.352461     -1.30115
    -1.390833     0.344040     -1.410259     -1.30115

Mean after scaling (should be ~0): [0.0, -0.0, -0.0, 0.0]
Std  after scaling (should be ~1): [1.0034, 1.0034, 1.0034, 1.0034]


## 7. TRAIN / TEST SPLIT ─────────────────────────────────────

In [8]:
print("\n" + "=" * 55)
print("STEP 7: Train / Test Split  (80% / 20%)")
print("=" * 55)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps class distribution equal
)

print(f"  X_train shape : {X_train.shape}")
print(f"  X_test  shape : {X_test.shape}")
print(f"  y_train shape : {y_train.shape}")
print(f"  y_test  shape : {y_test.shape}")

print("\nClass distribution in y_train:")
print(pd.Series(y_train).value_counts().sort_index()
        .rename(index=dict(enumerate(le.classes_))))

print("\nClass distribution in y_test:")
print(pd.Series(y_test).value_counts().sort_index()
        .rename(index=dict(enumerate(le.classes_))))


STEP 7: Train / Test Split  (80% / 20%)
  X_train shape : (120, 4)
  X_test  shape : (30, 4)
  y_train shape : (120,)
  y_test  shape : (30,)

Class distribution in y_train:
species_encoded
setosa        40
versicolor    40
virginica     40
Name: count, dtype: int64

Class distribution in y_test:
species_encoded
setosa        10
versicolor    10
virginica     10
Name: count, dtype: int64


## 8. SUMMARY ────────────────────────────────────────────────

In [9]:
print("\n" + "=" * 55)
print("PREPROCESSING COMPLETE – Summary")
print("=" * 55)
print(f"  Total samples    : {len(df)}")
print(f"  Features used    : {numerical_cols}")
print(f"  Target           : species (label-encoded)")
print(f"  Scaling method   : StandardScaler (Z-score)")
print(f"  Train samples    : {X_train.shape[0]}")
print(f"  Test  samples    : {X_test.shape[0]}")
print(f"\nReady for model training!")
print("=" * 55)


PREPROCESSING COMPLETE – Summary
  Total samples    : 150
  Features used    : ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
  Target           : species (label-encoded)
  Scaling method   : StandardScaler (Z-score)
  Train samples    : 120
  Test  samples    : 30

Ready for model training!
